# 요구사항 데이터셋 연구 EDA

동결된 `requirements_v0.3.0`을 대상으로 문서·요구사항 유형 분포, 입력 길이 편향, 문서 간 유사 문구, 위험 조건 표현과 40건 라벨링 파일럿 표본의 대표성을 점검합니다.

## 이 노트북에서 답하려는 질문

1. 특정 문서나 요구사항 유형이 데이터셋을 지나치게 지배하는가?
2. 문서별 입력 길이 차이가 모델 성능이나 비용 비교를 왜곡할 수 있는가?
3. 서로 다른 문서 사이에 유사 문구가 반복되어 문서 단위 평가에도 누수가 남을 수 있는가?
4. 40건 파일럿은 전체 품질을 추정할 대표 표본인가, 경계 사례 중심의 도전 표본인가?

## 해석 시 주의사항

- TF-IDF 유사도는 의미적 동일성을 보장하지 않습니다. 상위 쌍은 원문의 제공 주체·수량·상한·책임 조건을 사람이 비교해야 합니다.
- 위험 조건 표현은 정답 라벨이 아니라 후속 감사를 위한 후보 신호입니다.
- 대표 품질 추정용 표본과 경계·실패 분석용 표본은 구분해서 해석해야 합니다.

## 여기의 `위험 표현`은 blocker 범주가 아닙니다

아래 4절에서 쓰는 정규식 신호(`무상·추가`, `검수·성능기준`, `책임·배상` 등)는
**탐색용 표면 패턴**입니다. 라벨링 프롬프트가 쓰는 `blockers` 5범주와 이름이 겹치지만
정의가 다릅니다.

| | 이 노트북의 정규식 | 프롬프트의 `blockers` |
|---|---|---|
| 목적 | 감사 표본 층화용 후보 신호 | 입찰 전 확인이 필요한 조건 판정 |
| 판정 주체 | 문자열 매칭 | LLM (문맥 판단) |
| 예: `검수·성능기준` | `검수` 또는 `합격기준`이라는 **단어가 있으면** 해당 | **명시적 무제한 재작업** 또는 **구체적 수치 목표**가 있을 때만 해당 |

실제로 표준 조항 50건 실험에서 이 차이가 드러났습니다. 단어 유무로 판정하면
`기존 시스템 호환`, `가용성 보장` 같은 관행 조항이 전부 걸리는데, 실무 기준으로는
그냥 수용하는 조항입니다(결정 21, 26). 두 기준을 섞어 해석하면 안 됩니다.


In [ ]:
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.data.eda_requirements import normalize_requirement_type

DATASET_PATH = ROOT / 'data/processed/requirements_v0.3.0.jsonl'
PILOT_PATH = ROOT / 'data/samples/labeling_pilot_sample_v0.1.0.jsonl'
df = pd.read_json(DATASET_PATH, lines=True)
pilot = pd.read_json(PILOT_PATH, lines=True)

def compact_text(value):
    return re.sub(r'\s+', ' ', str(value)).strip()

for frame in (df, pilot):
    frame['normalized_type'] = frame['requirement_type'].apply(normalize_requirement_type)
    frame['analysis_text'] = frame['raw_requirement_text'].map(compact_text)
    frame['char_len'] = frame['analysis_text'].str.len()
    frame['word_len'] = frame['analysis_text'].str.split().str.len()

sns.set_theme(style='whitegrid')
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
font_candidates = ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'AppleGothic', 'DejaVu Sans']
selected_font = next(font for font in font_candidates if font in installed_fonts)
plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False
print(f'matplotlib font={selected_font}')
print(f'dataset={len(df):,} rows / {df.document_id.nunique()} documents')
print(f'pilot={len(pilot):,} rows / {pilot.document_id.nunique()} documents')

In [ ]:
display(Markdown('''## 1. 데이터 구성과 문서별 편향

문서별 행 수는 어떤 기관의 표현이 학습 신호를 많이 차지하는지 보여줍니다. 히트맵은 각 문서 내부의 유형 비율을 사용하므로 문서 크기가 달라도 구성 차이를 비교할 수 있습니다. 특정 유형이 한두 문서에 집중되면 행 무작위 분할에서 높은 점수가 나와도 새 기관으로 일반화되지 않을 수 있습니다.'''))

# 문서별 규모, 정규화 유형 분포와 문서×유형 편향
doc_counts = df['document_id'].value_counts().rename('count').to_frame()
doc_counts['ratio_pct'] = (doc_counts['count'] / len(df) * 100).round(1)
type_counts = df['normalized_type'].value_counts().rename('count').to_frame()
type_counts['ratio_pct'] = (type_counts['count'] / len(df) * 100).round(1)
display(doc_counts, type_counts)

doc_type = pd.crosstab(df['document_id'], df['normalized_type'], normalize='index') * 100
fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1, 2]})
sns.barplot(x=doc_counts['count'], y=doc_counts.index, ax=axes[0], color='#4C78A8')
axes[0].set(title='문서별 요구사항 수', xlabel='행 수', ylabel='document_id')
sns.heatmap(doc_type, cmap='YlGnBu', ax=axes[1], cbar_kws={'label': '문서 내 비율 (%)'})
axes[1].set(title='문서별 정규화 요구사항 유형 구성', xlabel='정규화 유형', ylabel='document_id')
plt.tight_layout()

dominant_type = doc_type.idxmax(axis=1).rename('dominant_type').to_frame()
dominant_type['ratio_pct'] = doc_type.max(axis=1).round(1)
display(dominant_type.sort_values('ratio_pct', ascending=False))
display(Markdown('**읽는 법:** `ratio_pct`가 큰 문서는 하나의 요구사항 유형에 편중된 문서입니다. 이후 모델 평가에서는 문서별 F1과 유형별 Recall을 함께 보고, 기관명 마스킹 실험도 고려해야 합니다.'))

In [ ]:
display(Markdown('''## 2. 입력 길이와 모델링 부담

본문 길이는 API 비용, 최대 입력 길이, 배치 내 패딩 낭비와 분류 난이도에 영향을 줍니다. 전체 분위수뿐 아니라 문서별 분포를 확인해야 특정 기관의 장문 서식이 모델 성능을 좌우하는지 판단할 수 있습니다. P95 이상 항목은 실제 모델 토크나이저 검사의 우선 후보입니다.'''))

# 입력 길이 분포와 문서별 장문 편향
length_summary = df[['char_len', 'word_len']].describe(percentiles=[.5, .9, .95, .99]).round(1)
doc_lengths = df.groupby('document_id')['char_len'].agg(['count', 'median', 'mean', 'max']).round(1)
doc_lengths['p95'] = df.groupby('document_id')['char_len'].quantile(.95).round(1)
display(length_summary, doc_lengths.sort_values('mean', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.histplot(df['char_len'], bins=40, ax=axes[0], color='#F58518')
for q, style in [(0.90, '--'), (0.95, ':'), (0.99, '-.')]:
    axes[0].axvline(df['char_len'].quantile(q), linestyle=style, label=f'P{int(q*100)}')
axes[0].set(title='요구사항 본문 문자 길이', xlabel='공백 정규화 후 문자 수')
axes[0].legend()
order = doc_lengths.sort_values('median').index
sns.boxplot(data=df, x='char_len', y='document_id', order=order, showfliers=False, ax=axes[1], color='#72B7B2')
axes[1].set(title='문서별 본문 길이 분포', xlabel='문자 수', ylabel='document_id')
plt.tight_layout()

long_cut = df['char_len'].quantile(.95)
long_items = df.loc[df['char_len'] >= long_cut, ['requirement_uid', 'document_id', 'normalized_type', 'char_len', 'requirement_name']]
display(long_items.sort_values('char_len', ascending=False).head(20))
display(Markdown('**주의:** 문자 수는 모델 토큰 수가 아닙니다. 모델이 확정되면 해당 토크나이저 기준으로 P95·P99와 실제 절단 비율을 다시 계산해야 합니다.'))

In [ ]:
display(Markdown('''## 3. 문서 간 반복 문구와 평가 누수 후보

정확히 같은 본문은 표준 조항의 중복 여부를 보여줍니다. 문자 n-gram TF-IDF는 표현이 조금 달라진 유사 조항도 찾습니다. 같은 문서 내부의 쌍은 제외하여 문서 단위 분할에서도 학습·평가 문서 사이에 남을 수 있는 반복 표현을 확인합니다.'''))

# 정확히 같은 본문과 문서 간 TF-IDF 유사 문구 탐색
exact_duplicates = (
    df.groupby('analysis_text')
      .filter(lambda group: len(group) > 1)
      .sort_values('analysis_text')
      [['requirement_uid', 'document_id', 'requirement_name', 'analysis_text']]
)
print(f'exact duplicate rows: {len(exact_duplicates):,}')
display(exact_duplicates.drop(columns='analysis_text').head(30))

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=30_000, sublinear_tf=True)
matrix = vectorizer.fit_transform(df['analysis_text'])
similarity = cosine_similarity(matrix)
same_document = df['document_id'].to_numpy()[:, None] == df['document_id'].to_numpy()[None, :]
similarity[same_document] = -1.0

nearest_index = similarity.argmax(axis=1)
nearest_score = similarity[np.arange(len(df)), nearest_index]
pairs = []
seen = set()
for left, right, score in zip(range(len(df)), nearest_index, nearest_score):
    key = tuple(sorted((left, int(right))))
    if key in seen:
        continue
    seen.add(key)
    pairs.append({
        'similarity': round(float(score), 3),
        'left_uid': df.iloc[left]['requirement_uid'],
        'right_uid': df.iloc[right]['requirement_uid'],
        'left_name': df.iloc[left]['requirement_name'],
        'right_name': df.iloc[right]['requirement_name'],
    })
cross_document_pairs = pd.DataFrame(pairs).sort_values('similarity', ascending=False)
display(cross_document_pairs.head(30))
display(Markdown('**읽는 법:** 유사도 상위 쌍부터 원문을 대조하되 라벨이 같아야 한다고 가정하지 않습니다. 제공 주체, 무상 범위, 수량 상한, 검수 기준과 책임 주체가 다르면 유사 문구라도 다른 조치가 타당할 수 있습니다.'))

In [ ]:
display(Markdown('''## 4. 위험 표현 후보와 파일럿 대표성

위험 표현 빈도는 감사 표본을 층화하는 보조 정보입니다. 이어지는 분포 차이는 40건 파일럿이 전체 1,024건을 얼마나 닮았는지 보여줍니다. 절대 비율 차이가 큰 범주는 파일럿 결과를 전체 품질로 일반화할 때 특히 주의해야 합니다.'''))

# 위험 조건 표현의 빈도와 40건 파일럿 표본 대표성
surface_signal_patterns = {
    '무상·추가_표면신호': r'무상|추가.{0,8}(?:개발|제공|지원)',
    '협의·추후확정_표면신호': r'협의|추후.{0,6}확정|별도.{0,6}정함',
    '검수·성능_표면신호': r'검수|합격.{0,6}기준|성능.{0,6}(?:기준|보장)',
    '책임·배상_표면신호': r'책임|배상|지체상금|귀책',
    '상주·인력_표면신호': r'상주|투입.{0,5}인력|전담.{0,5}인력',
    '수량·상한_표면신호': r'이상|이하|이내|최대|최소|[0-9]+\s*(식|명|개|대|TB)',
}
for name, pattern in surface_signal_patterns.items():
    df[name] = df['analysis_text'].str.contains(pattern, regex=True, na=False)
risk_summary = pd.DataFrame({
    'count': df[list(surface_signal_patterns)].sum(),
    'ratio_pct': (df[list(surface_signal_patterns)].mean() * 100).round(1),
}).sort_values('ratio_pct', ascending=False)
display(risk_summary)

def distribution_gap(population, sample, column):
    pop = population[column].value_counts(normalize=True)
    sam = sample[column].value_counts(normalize=True)
    result = pd.concat([pop.rename('dataset'), sam.rename('pilot')], axis=1).fillna(0) * 100
    result['abs_gap_pp'] = (result['pilot'] - result['dataset']).abs()
    return result.sort_values('abs_gap_pp', ascending=False).round(1)

document_gap = distribution_gap(df, pilot, 'document_id')
type_gap = distribution_gap(df, pilot, 'normalized_type')
length_compare = pd.DataFrame({
    'dataset': df['char_len'].quantile([.5, .9, .95, .99]),
    'pilot': pilot['char_len'].quantile([.5, .9, .95, .99]),
}).round(1)
display(document_gap, type_gap, length_compare)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
document_gap[['dataset', 'pilot']].sort_values('dataset').plot.barh(ax=axes[0], title='문서 분포: 전체 vs 파일럿')
type_gap[['dataset', 'pilot']].sort_values('dataset').plot.barh(ax=axes[1], title='유형 분포: 전체 vs 파일럿')
for ax in axes:
    ax.set_xlabel('비율 (%)')
plt.tight_layout()
display(Markdown('''### 다음 판단으로 연결하기

- 문서·유형 비율 차이가 크면 현재 40건은 **도전 표본**으로 명시하고 전체 품질 추정용 층화 무작위 표본을 별도로 만듭니다.
- 장문 비율이 다르면 모델 비교 시 길이 구간별 성능과 비용을 따로 보고합니다.
- 위험 표현은 목표 라벨 분포를 강제로 맞추는 용도가 아니라 경계 사례가 빠지지 않도록 감사 표본을 보강하는 용도로만 사용합니다.'''))